# The four-dataset confound diagnostic with Segment Anything masks (RQ3)

The original diagnostic (Section IV-F) used the Otsu mask, which returns the whole frame on all four external datasets,
so its controls were degenerate there. This notebook replaces **only the mask** with Segment Anything (ViT-B,
centre-point prompt, as in the field evaluation) and keeps everything else identical to
`scripts/original/cross_dataset_confound.py`: the same datasets, the same 200 images per class (`random.Random(0)`),
the same DINO attention score, the same three controls.

**Positive control first.** On TeaLeafDiseaseBD, where the confound is known, SAM must recover it. Pre-declared
criteria: healthy/diseased area ratio ≥ 1.5, shape-only AUC ≥ 0.75, attention collapse ≥ 0.20 (Otsu gave 1.94, 0.83,
0.39). If SAM fails these, the external results are reported as uninterpretable.

**Mask gate.** A SAM mask is accepted only if it covers 3–90% of the frame; a dataset is scored only if ≥ 70% of its
images get an accepted mask (the original script's rule). Contact sheets with mask outlines are saved for every dataset.

Output: `results_revision_v1/sam_confound.json` + `sam_contact_*.png` — upload them. Runtime ~30–45 min on a GPU.

In [ ]:
# CELL 0 — CONFIG
DRIVE      = '/content/drive/MyDrive'
TEA_ROOT   = f'{DRIVE}/Tea Leaf Dataset'
STUDY      = f'{DRIVE}/TeaLeaf_DINOv2_Study'
RES        = f'{STUDY}/results_revision_v1'
CVD        = f'{STUDY}/cross_validation_datasets'
OUT        = f'{RES}/sam_confound.json'
LOCAL_DATA = '/tmp/tea_leaf_local'
N_PER_CLASS, SEED = 200, 0              # as in the original run (notebook 08)
SAM_MAX_SIDE, SAM_AREA = 1024, (0.03, 0.90)
MIN_MASK_RATE = 0.70
PLANTDOC_HEALTHY = ["apple leaf", "bell_pepper leaf", "blueberry leaf", "cherry leaf",
                    "raspberry leaf", "soyabean leaf", "tomato leaf", "grape leaf"]
DATASETS = {"TeaLeafDiseaseBD": (LOCAL_DATA, ("healthy",)),
            "PlantVillage": (f"{CVD}/PlantVillage", ("healthy",)),
            "RiceLeafBD": (f"{CVD}/RiceLeafDiseaseBD/Original images", ("healthy",)),
            "RoCoLe": (f"{CVD}/RoCoLe_binary", ("healthy",)),
            "PlantDoc (field)": (f"{CVD}/PlantDoc", tuple(PLANTDOC_HEALTHY))}
# the published Otsu results (artifacts/cross_dataset_confound.json), for the old-vs-new table
OTSU = {"TeaLeafDiseaseBD": (1.936, 0.834, 0.928, 0.543), "PlantVillage": (1.018, 0.574, 0.611, 0.574),
        "RiceLeafBD": (1.060, 0.623, 0.730, 0.719), "RoCoLe": (0.922, 0.572, 0.727, 0.750), "PlantDoc (field)": (1.074, 0.600, 0.644, 0.657)}


In [ ]:
# CELL 1 — setup, SAM checkpoint, original modules, local copy of TeaLeafDiseaseBD
from google.colab import drive; drive.mount('/content/drive')
!pip -q install timm git+https://github.com/facebookresearch/segment-anything.git
!wget -q -nc --no-check-certificate https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P /content/
import os, sys, io, json, glob, time, random, shutil, hashlib, warnings
import numpy as np, cv2, torch, timm, torchvision.transforms as T
from PIL import Image, ImageDraw
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device', device)
for m in ('shape_confound.py',):
    hit = glob.glob(f'{STUDY}/**/{m}', recursive=True); assert hit, m + ' not found on Drive'
    shutil.copy(hit[0], f'/content/{m}')
sys.path.insert(0, '/content'); import shape_confound as sc
assert hasattr(sc, 'leaf_mask') and hasattr(sc, 'shape_features') and hasattr(sc, 'attn_scores')
def _cp(t):
    if not os.path.exists(t[1]): os.makedirs(os.path.dirname(t[1]), exist_ok=True); shutil.copyfile(*t)
tasks = [(os.path.join(r, f), os.path.join(LOCAL_DATA, os.path.relpath(r, TEA_ROOT), f)) for r, _, fs in os.walk(TEA_ROOT) for f in fs]
with ThreadPoolExecutor(32) as ex: list(ex.map(_cp, tasks))
print('TeaLeafDiseaseBD local copy:', len(tasks), 'files')


In [ ]:
# CELL 2 — DINO ViT-S/16 with the CLS-attention hook (verbatim from notebook 08)
dv1_path = os.path.join(STUDY, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    os.system('wget -q -O "%s" https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth' % dv1_path)
dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
dino_v1.load_state_dict({k.replace('module.', '').replace('backbone.', '').replace('encoder.', ''): v for k, v in ckpt.items() if 'head' not in k}, strict=False)
dino_v1.eval().to(device)
_dv1_store = {}; att = dino_v1.blocks[-1].attn
def _fwd(x, **kw):
    B, N, Cc = x.shape
    q, k, v = att.qkv(x).reshape(B, N, 3, att.num_heads, Cc // att.num_heads).permute(2, 0, 3, 1, 4).unbind(0)
    s = (Cc // att.num_heads) ** -0.5
    _dv1_store['cls_attn'] = ((q[:, :, 0:1, :] @ k.transpose(-2, -1)) * s).softmax(-1)[:, :, 0, 1:].detach().cpu()
    x = (((q @ k.transpose(-2, -1)) * s).softmax(-1) @ v).transpose(1, 2).reshape(B, N, Cc)
    return att.proj_drop(att.proj(x))
att.forward = _fwd
tf_dv1 = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
DV1_GRID = 14; topk = max(1, int(0.1 * DV1_GRID * DV1_GRID))
with torch.no_grad(): _ = dino_v1(torch.randn(1, 3, 224, 224).to(device))
print('attention hook OK', _dv1_store['cls_attn'].shape)


In [ ]:
# CELL 3 — SAM, centre-point prompt (same recipe as the field evaluation)
from segment_anything import sam_model_registry, SamPredictor
_h = hashlib.md5(open('/content/sam_vit_b_01ec64.pth', 'rb').read()).hexdigest()
assert _h.startswith('01ec64'), 'SAM checkpoint checksum mismatch'
predictor = SamPredictor(sam_model_registry['vit_b'](checkpoint='/content/sam_vit_b_01ec64.pth').to(device).eval())
def sam_mask(rgb):
    H, W = rgb.shape[:2]; s = min(1.0, SAM_MAX_SIDE / max(H, W))
    small = cv2.resize(rgb, (int(W * s), int(H * s)), interpolation=cv2.INTER_AREA) if s < 1 else rgb
    predictor.set_image(small); h, w = small.shape[:2]
    masks, scores, _ = predictor.predict(point_coords=np.array([[w / 2, h / 2]]), point_labels=np.array([1]), multimask_output=True)
    best, bs = None, -1
    for m, q in zip(masks, scores):
        if SAM_AREA[0] <= m.mean() <= SAM_AREA[1] and q > bs: best, bs = m, q
    return best                                     # None = no plausible leaf mask


In [ ]:
# CELL 4 — the diagnostic with the mask swapped (all else as scripts/original/cross_dataset_confound.py)
def sample(root, hk):
    rng = random.Random(SEED); recs = []
    classes = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    for c in classes:
        fs = [f for f in os.listdir(os.path.join(root, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for f in rng.sample(fs, min(N_PER_CLASS, len(fs))):
            recs.append(dict(path=os.path.join(root, c, f), cls=c, y=0 if any(k in c.lower() for k in hk) else 1))
    return recs
def contact(name, items):
    tiles = []
    for img, m, c in items[:16]:
        im = img.resize((224, 224)).convert('RGB'); d = ImageDraw.Draw(im)
        if m is not None:
            mm = cv2.resize(m.astype(np.uint8), (224, 224), interpolation=cv2.INTER_NEAREST)
            for cnt in cv2.findContours(mm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]:
                d.line([tuple(p[0]) for p in cnt] + [tuple(cnt[0][0])], fill=(255, 0, 255), width=3)
        else: d.text((6, 6), 'no mask', fill=(255, 0, 0))
        d.text((6, 206), c[:28], fill=(255, 255, 0)); tiles.append(im)
    sheet = Image.new('RGB', (4 * 224, 4 * 224), 'white')
    for i, t in enumerate(tiles): sheet.paste(t, ((i % 4) * 224, (i // 4) * 224))
    p = f'{RES}/sam_contact_{name.split()[0]}.png'; sheet.save(p); return p
def scan(name, root, hk):
    recs = sample(root, hk); S, G, Wv, A, Y, IOU, shown = [], [], [], [], [], [], []
    t0 = time.time(); order = np.random.default_rng(1).permutation(len(recs))
    for n_i, k in enumerate(order):
        r = recs[k]
        try: img = Image.open(r['path']).convert('RGB')
        except Exception: continue
        rgb = np.array(img); m = sam_mask(rgb)
        if len(shown) < 16: shown.append((img, m, r['cls']))
        if m is None: continue
        f = sc.shape_features(m)
        if f is None: continue
        g, w = sc.attn_scores(img, dino_v1, tf_dv1, _dv1_store, device, DV1_GRID, [0, 1], topk, m)
        if name == 'TeaLeafDiseaseBD':                       # agreement with the Otsu mask where Otsu is reliable
            mo = sc.leaf_mask(img)
            if mo is not None:
                ms = cv2.resize(m.astype(np.uint8), mo.shape[::-1], interpolation=cv2.INTER_NEAREST).astype(bool)
                IOU.append(float((ms & mo).sum() / max(1, (ms | mo).sum())))
        S.append(f); G.append(g); Wv.append(w); A.append(float(m.mean())); Y.append(r['y'])
    rate = len(Y) / len(recs); sheet = contact(name, shown)
    row = dict(dataset=name, n_sampled=len(recs), n_scored=len(Y), mask_rate=rate, contact_sheet=sheet, minutes=(time.time() - t0) / 60)
    if rate < MIN_MASK_RATE or len(set(Y)) < 2:
        row['scored'] = False; print('%-18s mask rate %.0f%% -> NOT scored' % (name, 100 * rate)); return row
    S, G, Wv, A, Y = map(np.array, (S, G, Wv, A, Y)); ok = ~np.isnan(Wv); dir_ = lambda a: max(a, 1 - a)
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    shape = max(cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), S, Y, cv=cv, scoring='roc_auc').mean(),
                cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1), S, Y, cv=cv, scoring='roc_auc').mean())
    naive = dir_(roc_auc_score(Y, G)); within = dir_(roc_auc_score(Y[ok], Wv[ok]))
    row.update(scored=True, healthy_area=float(A[Y == 0].mean()), diseased_area=float(A[Y == 1].mean()),
               area_ratio=float(A[Y == 0].mean() / A[Y == 1].mean()), area_auc=float(dir_(roc_auc_score(Y, A))),
               shape_only_auc=float(shape), naive_attention_auc=float(naive), within_leaf_attention_auc=float(within),
               attention_collapse=float(naive - within), median_mask_area=float(np.median(A)),
               iou_with_otsu=(dict(median=float(np.median(IOU)), p10=float(np.percentile(IOU, 10)), n=len(IOU)) if IOU else None))
    print('%-18s mask %.0f%% | area ratio %.2f | shape-only %.3f | naive %.3f -> within %.3f (collapse %+.3f) | median mask %.2f | %.1f min'
          % (name, 100 * rate, row['area_ratio'], shape, naive, within, naive - within, row['median_mask_area'], row['minutes']))
    return row


In [ ]:
# CELL 5 — POSITIVE CONTROL: SAM must recover the known confound on TeaLeafDiseaseBD
ROWS = [scan('TeaLeafDiseaseBD', *DATASETS['TeaLeafDiseaseBD'])]
pc = ROWS[0]
VALID = bool(pc.get('scored') and pc['area_ratio'] >= 1.5 and pc['shape_only_auc'] >= 0.75 and pc['attention_collapse'] >= 0.20)
print('SAM vs Otsu mask on our data: median IoU %s' % (pc.get('iou_with_otsu'),))
print('INSTRUMENT VALID (recovers the known confound):', VALID)
if not VALID: print('-> SAM does not recover the confound on our data; the external results below cannot be interpreted.')


In [ ]:
# CELL 6 — the four external datasets, then old (Otsu) vs new (SAM)
for name in [k for k in DATASETS if k != 'TeaLeafDiseaseBD']:
    root, hk = DATASETS[name]
    if not os.path.isdir(root): print(name, 'NOT FOUND at', root); continue
    ROWS.append(scan(name, root, hk))
print('\n%-18s %18s %18s %26s' % ('dataset', 'area ratio O/S', 'shape-only O/S', 'collapse O/S'))
for r in ROWS:
    o = OTSU[r['dataset']]
    if r.get('scored'):
        print('%-18s %8.2f / %-7.2f %8.3f / %-7.3f %10.3f / %-+7.3f' % (r['dataset'], o[0], r['area_ratio'], o[1], r['shape_only_auc'], o[2] - o[3], r['attention_collapse']))
    else:
        print('%-18s not scored (mask rate %.0f%%)' % (r['dataset'], 100 * r['mask_rate']))
json.dump(dict(instrument_valid=VALID, criteria=dict(area_ratio=1.5, shape_only=0.75, collapse=0.20), rows=ROWS,
               otsu_published=OTSU, sam=dict(model='vit_b', prompt='centre point', max_side=SAM_MAX_SIDE, area=SAM_AREA)),
          open(OUT, 'w'), indent=1, default=float)
print('saved', OUT, '+ sam_contact_*.png -> upload them')
